# H SC 790 · Lab 4 · Two Front Doors, One Registry

**Week 5.** Due Thursday, February 25, 11:59 PM · 30 points.

**What earns full credit**
- Runs end to end from a fresh runtime
- Both sources land in one schema
- Every FHIR page was read; rows balance
- AI-use note included

**Rules.** Everything here is synthetic. Never load protected health information or confidential employer data into Colab or GitHub. The lab is finished when it runs top to bottom from a fresh runtime (Runtime → Restart session and run all).

The data dictionary for this lab is the README in the lab's folder of the course repository.

Cells marked `# YOUR CODE HERE` and *Your answer here.* are yours to complete. Add cells freely.

In [ ]:
# Setup: run this cell first, every time.
import os, json, urllib.request
import numpy as np
import pandas as pd

# The course data, read straight from the course repository. Never from your own machine.
COURSE_URL = os.environ.get("HSC790_DATA", "https://raw.githubusercontent.com/lmlevine/hsc790-spr27/main/data/")

def read_text(rel):
    """Text of one file in the course data folder, for example read_text('SNAPSHOT.txt')."""
    return urllib.request.urlopen(COURSE_URL + rel).read().decode()

print("Data snapshot:", read_text("SNAPSHOT.txt").strip())

Laboratory results reach the registry through two channels: a batch of HL7 v2 messages and a FHIR server. Parse both, map them into one staging layout, apply your checks, and compare the channels.

The FHIR server here is fictional, so this notebook reads the pages it returned from the course repository through `fhir_get`, which stands in for `requests.get(url, params=params).json()`. If your instructor announces a live test server, set `BASE` to it and use `requests` instead.

In [ ]:
from urllib.parse import urlparse, parse_qs

text = read_text("lab04/elr_batch_2027-02-15.hl7")

BASE = "https://fhir.puente-hio.example/baseR4"
TAG = "https://hsc790.example/tags|hsc790-spring2027"

def fhir_get(url, params=None):
    """The response the server gave to GET url: the capability statement, or one page of the search."""
    if urlparse(url).path.endswith("/metadata"):
        return json.loads(read_text("lab04/fhir/metadata.json"))
    page = parse_qs(urlparse(url).query).get("_page", ["1"])[0]
    return json.loads(read_text(f"lab04/fhir/observation_page_{page}.json"))

print(text[:600])

## 1. Parse one message by hand

Split the batch into messages (each starts with `MSH`), then split one message into segments, fields, and components. Print PID-5, OBX-3, and OBX-5. The standard separates segments with carriage returns; files often arrive with newlines, so handle both.

In [ ]:
lines = text.replace("\r", "\n").split("\n")
messages, cur = [], []
for line in lines:
    if line.startswith("MSH|") and cur:
        messages.append("\n".join(cur)); cur = []
    if line:
        cur.append(line)
messages.append("\n".join(cur))
print(len(messages), "messages")

msg = messages[0]
# --- HL7 v2: parse one message by hand (slide) ---
segs = [s.split('|') for s in
        msg.replace('\n', '\r').split('\r') if s]
pid = next(s for s in segs if s[0] == 'PID')
family, given = pid[5].split('^')[:2]
for obx in (s for s in segs if s[0] == 'OBX'):
    test = obx[3].split('^')     # code ^ text ^ system
    value = obx[5].split('^')
print("PID-5:", family, given, "| OBX-3:", test, "| OBX-5:", value)

## 2. Parse the whole batch

Write a function that turns one message into a row (or rows) with the fields the registry needs: sender, message time and control id, accession number, patient name, birth date, sex, race, ethnicity, address, test code and its coding system, result, result status, specimen date. Remember that in `MSH` the field separator is itself MSH-1, so `split('|')[n]` is MSH-(n+1).

In [ ]:
def parse_message(msg):
    # YOUR CODE HERE
    pass

v2 = pd.DataFrame([parse_message(m) for m in messages])
print(len(v2), "rows from", len(messages), "messages")
v2.head(3).T

## 3. Query the FHIR server

Read the capability statement. Search for Observations with patients included. Follow every page.

In [ ]:
cap = fhir_get(BASE + "/metadata")
print(cap["resourceType"], cap["fhirVersion"], [r["type"] for r in cap["rest"][0]["resource"]])

# --- FHIR: search and follow every page (slide) ---
url, params = BASE + '/Observation', {
    '_tag': TAG, '_count': 100,
    '_include': 'Observation:patient'}
resources = []
while url:
    b = fhir_get(url, params=params)
    params = None        # next link is complete
    resources += [e['resource'] for e in b.get('entry', [])]
    url = next((l['url'] for l in b.get('link', [])
                if l['relation'] == 'next'), None)
pd.Series([r["resourceType"] for r in resources]).value_counts()

## 4. Map both into staging

One mapping per source, into the same columns as your v2 rows. Keep the source system and the original codes alongside the mapped ones. A Patient included on several pages is one patient.

In [ ]:
# YOUR CODE HERE

## 5. Run your checks and compare the channels

Apply the checks from Lab 2 that fit: completeness of demographics, conformance of codes. Compare the two channels, and break the v2 channel down by sender.

In [ ]:
# YOUR CODE HERE

## 6. Handle status and corrections

Show that an amended or entered-in-error result does not become a second case. Find every accession number that appears more than once in either channel, decide the rule, apply it, and show the row counts before and after.

In [ ]:
# YOUR CODE HERE

## Interpretation

Cite your own output in each answer.

**1. What share of v2 results had no LOINC?** By sending facility. What would you ask that laboratory to change?

*Your answer here.*

**2. Which channel was more complete?** Compare race, address, and coded results between v2 and FHIR.

*Your answer here.*

**3. How did you treat status and corrections?** Show one example and the rule you applied.

*Your answer here.*

**4. What would onboarding this sender take?** List the steps, the people on each side, and your estimate of elapsed time.

*Your answer here.*

## AI-use note (required)

Say which AI tools you used, for what, and one thing you checked or corrected in their output. "None" is an acceptable answer.

*Your note here.*

## Submit

1. Runtime → Restart session and run all. Fix anything that fails.
2. File → Save a copy in GitHub, into `labs/lab04/` of your course repository.